# Extended Thinking (Reasoning)

---

## What It Is

Extended thinking gives Claude a **reasoning scratchpad** before generating its final
response — like working through a problem on scratch paper before writing the answer.

Often referred to as **"reasoning"** in the Anthropic documentation.

---

## Response Structure With Thinking Enabled

Without thinking, the assistant message has one block:
```
Assistant Message → TextBlock (response)
```

With thinking enabled, it has two blocks:
```
Assistant Message → ThinkingBlock  ("I should cover what recursion is, how it works...")
                  → TextBlock      ("Recursion is a powerful programming technique...")
```

---

## The Thinking Block — Two Variants

### Visible Thinking
```json
{
  "type": "thinking",
  "thinking": "The user is asking me to...",
  "signature": "EqoBCkgIAhABGAIiQDbvHj8yA..."
}
```

### Redacted Thinking
```json
{
  "type": "redacted_thinking",
  "data": "kgIAh/ABGAIiQDbvHj8yAQDbvHj8yA..."
}
```

Redacted thinking occurs when Claude's reasoning is flagged by internal safety systems.
The actual thinking is stored in **encrypted form** — you cannot read it, but you
must pass it back to Claude in follow-up requests to preserve conversation context.

---

## The Signature

A **cryptographic token** attached to every visible thinking block.

- Verifies the thinking text was generated by Claude and was not modified
- Prevents developers from tampering with the reasoning process
- Required when sending thinking blocks back to Claude in multi-turn conversations

---

## Trade-offs

| Benefit | Cost |
|---|---|
| Better reasoning on complex tasks | Higher token cost (you pay for thinking tokens) |
| Higher accuracy on hard problems | Increased latency |
| Transparency into reasoning process | More complex response handling in code |

---

## When to Use It

> Run evals first. Enable extended thinking only if standard prompting — after
> thorough prompt engineering — still doesn't meet your accuracy requirements.

**Decision flow:**
```
Write prompt → optimise with prompt engineering → run evals
      |
      Is accuracy sufficient?
      Yes → ship it
      No  → enable extended thinking → re-eval
```

---

## Implementation

Update `chat()` to accept thinking parameters:
```python
def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024       # minimum is 1024 tokens
):
    params = { ... }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget": thinking_budget
        }

    message = client.messages.create(**params)
    return message
```

**Important:** `max_tokens` must be **greater than** `thinking_budget`.

### Usage
```python
# Standard call
chat(messages)

# With extended thinking
chat(messages, thinking=True, thinking_budget=2048)
```

---

## Compatibility Note

> Extended thinking is **not compatible** with some features, including
> message pre-filling and temperature settings.
> Full list: https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking#feature-compatibility

---

## Key Takeaways

- Thinking blocks come **before** the text block in the content list
- Always pass thinking blocks (including redacted ones) back to Claude in follow-up turns
- Never modify the thinking text — the signature will invalidate it
- Use `budget_tokens` to control reasoning depth and cost
- Default to standard prompting first — use thinking only when evals show it's needed

# Vision — Working with Images

---

## Image Handling Limits

| Constraint | Limit |
|---|---|
| Max images per request | 100 (across all messages) |
| Max size per image | 5 MB |
| Max dimensions (single image) | 8000 × 8000 px |
| Max dimensions (multiple images) | 2000 × 2000 px |
| Supported input formats | Base64 encoding or URL |
| Token cost formula | `(width px × height px) / 750` |

---

## Message Structure
```
User Message → Image Block  (raw image data)
             → Text Block   ("What do you see in this image?")

Assistant Message → Text Block ("I see a tall tree...")
```

The conversation flow is identical to text-only interactions.

---

## Sending an Image (Base64)
```python
import base64

with open("image.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "What do you see in this image?"
    }
])
```

---

## Prompting Techniques for Images

All text prompting techniques apply to images. Simple prompts lead to poor results.

**Simple prompt (wrong answer):**
```
"How many marbles are in this image?"
→ Claude returns 13  ✗  (correct answer is 12)
```

**Structured step-by-step prompt (correct answer):**
```
Analyze this image of marbles and determine the exact count using this methodology:
1. Begin by identifying each unique marble one at a time. Assign each a number.
2. Verify your result by counting with a different method. Start from the
   bottom-left corner and work row by row, from left to right.

What is the exact, verified number of marbles in this image?
→ Claude returns 12  ✓
```

### Techniques that improve accuracy

- **Step-by-step guidelines** — break analysis into explicit numbered steps
- **One-shot examples** — include a reference image with a known answer before
  asking about the target image
- **Verification steps** — ask Claude to confirm its answer using a second method

---

## One-Shot Example Pattern
```
User Message:
    [Image 1 — reference]
    "The image above has 11 marbles in it."
    [Image 2 — target]
    "How many marbles are in this image?"

→ Claude answers correctly by using Image 1 as a calibration reference
```

---

## Real-World Example — Fire Risk Assessment

**Problem:** Insurance companies need to assess fire risk around residences.
Sending a physical inspector to every property is expensive.

**Solution:** Use high-resolution satellite imagery + Claude.

**What Claude looks for:**
- Dense, close-packed trees near the residence
- Difficult access routes for emergency services
- Branches overhanging the roof

**Prompt structure (abbreviated):**
```
1. Residence identification — locate the primary structure
2. Tree overhang analysis — estimate % of roof covered by canopy
3. Fire risk assessment — evaluate wildfire vulnerability
4. Defensible space — identify continuous fuel paths
5. Fire risk rating — assign 1 (Low) to 4 (Severe)

For each item, write one sentence. Final response = numerical rating.
```

This structured prompt produces far more reliable ratings than simply asking
"provide a fire risk score."

---

## Key Takeaway

> The same prompt engineering principles that improve text responses
> also improve image analysis.  
> Invest in detailed, structured prompts with explicit steps and examples
> rather than relying on simple one-line questions.

# PDF Processing

---

## How It Works

PDF processing works almost identically to image processing — the only differences
are the block type, media type, and file extension.

---

## Code
```python
import base64

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",           # <-- "document" not "image"
            "source": {
                "type": "base64",
                "media_type": "application/pdf",   # <-- PDF media type
                "data": file_bytes,
            },
        },
        {
            "type": "text",
            "text": "Summarize the document in one sentence"
        },
    ],
)

chat(messages)
```

---

## Image vs PDF — What Changes

| Field | Image | PDF |
|---|---|---|
| File extension | `.png` / `.jpg` | `.pdf` |
| Block `type` | `"image"` | `"document"` |
| `media_type` | `"image/png"` | `"application/pdf"` |
| Variable name | `image_bytes` | `file_bytes` (convention) |

---

## What Claude Can Extract from PDFs

- Text content throughout the document
- Images and charts embedded in the PDF
- Tables and their data relationships
- Document structure and formatting

Claude handles all content types within a PDF in a single request —
no separate extraction pipeline needed.

---

## Key Takeaway

> PDF processing is a minimal adaptation of the image processing pattern.
> Swap the block type to `"document"`, update the media type to `"application/pdf"`,
> and Claude handles the rest — including embedded images, tables, and structure.

# PDF Citations

---

## What Citations Do

When Claude answers from a document, citations create a **traceable link** from each
claim back to the exact location in your source material — showing users where
information actually came from.

---

## Enabling Citations

Add `title` and `citations` fields to your document block:
```python
{
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": file_bytes,
    },
    "title": "earth.pdf",                  # readable name for the document
    "citations": { "enabled": True }       # enable citation tracking
}
```

---

## Citation Structure

When citations are enabled, Claude's response includes structured citation data
alongside its text:

| Field | Example | Purpose |
|---|---|---|
| `cited_text` | "Earth's atmosphere and oceans were formed by volcanic activity..." | Exact text from the document supporting the claim |
| `document_index` | `0` | Which document (useful when multiple are provided) |
| `document_title` | `"earth.pdf"` | Title assigned to the document |
| `start_page_number` | `4` | Page where cited text begins |
| `end_page_number` | `5` | Page where cited text ends |

---

## Citations with Plain Text

Citations also work on plain text sources — swap `base64`/`application/pdf` for `text`:
```python
{
    "type": "document",
    "source": {
        "type": "text",
        "media_type": "text/plain",
        "data": article_text,
    },
    "title": "earth_article",
    "citations": { "enabled": True }
}
```

With plain text, instead of page numbers you get **character positions** pinpointing
exactly where in the text the cited content appears.

---

## Building UIs with Citations

Citations enable interactive interfaces where users can:
- Hover over citation markers to see the exact source text
- Click through to the page in the original document
- Verify that Claude's answer is grounded in actual source material

---

## When to Use Citations

- Users need to **verify information** for accuracy
- You're working with **authoritative documents** users should be able to reference
- **Transparency** about sources is critical to your application
- Users may want to explore the **broader context** around specific facts

---

## Key Takeaway

> Citations transform Claude from a black box into a transparent research assistant
> that shows its work.  
> Enable them by adding `"title"` and `"citations": {"enabled": True}` to your
> document block — no other changes needed.

# Prompt Caching

---

## What It Is

Caching saves the **preprocessing work** Claude does on your prompt so it can be
reused on follow-up requests — making them faster and cheaper.

### What Claude does on every request (without caching):
```
Your text → Tokenize → Embed → Add context → [discarded after response]
```

### With caching:
```
Your text → Tokenize → Embed → Add context → [stored in cache for 1 hour]
                                                     ↓
                                   Follow-up request → reuse cached work
```

---

## Cost Savings — Not Fewer Tokens, Just Cheaper Ones

The same tokens are always sent. You just pay less for the cached portion.

| | Token count sent | Price paid |
|---|---|---|
| Without cache | 10,000 | 100% |
| Cache write (first request) | 10,000 | ~105% |
| Cache read (follow-up requests) | 10,000 | ~10% |

**Example — 10,000-token document, 3 questions:**
```
Without caching:  3 × 10,000 = 30,000 tokens @ full price
With caching:     10,000 (write) + 2 × 1,000 (reads) ≈ 85% cost reduction
```

---

## Cache Breakpoints

Caching is **not automatic** — you must manually add a `cache_control` field to a block.

- Everything **before and including** the breakpoint gets cached
- Content **after** the breakpoint is processed normally
- Cache is only reused if the content up to the breakpoint is **identical**
- Up to **4 cache breakpoints** allowed per request
- Minimum cacheable content: **1024 tokens**

### Shorthand vs Longhand block syntax
```python
# Shorthand — no room for cache_control
{"role": "user", "content": "Hi there!"}

# Longhand — required when adding cache_control
{
    "role": "user",
    "content": [
        {
            "type": "text",
            "text": "<Long prompt>",
            "cache_control": {"type": "ephemeral"}   # <-- cache breakpoint
        }
    ]
}
```

---

## Where to Place Breakpoints

Cache breakpoints are not restricted to text blocks:

| Location | When to use |
|---|---|
| **System prompt** | Long agent instructions that don't change per request |
| **Tool definitions** | Large tool schemas sent on every agentic loop iteration |
| **Messages** | Large documents being queried multiple times |
| Image blocks, tool results | Any large static content |

System prompts and tool definitions are the **most common caching opportunities**
because they rarely change between requests.

---

## Implementation

### Caching tool schemas (add to last tool only)
```python
if tools:
    tools_clone = tools.copy()
    last_tool = tools_clone[-1].copy()
    last_tool["cache_control"] = {"type": "ephemeral"}
    tools_clone[-1] = last_tool
    params["tools"] = tools_clone
```

### Caching the system prompt
```python
if system:
    params["system"] = [
        {
            "type": "text",
            "text": system,
            "cache_control": {"type": "ephemeral"}
        }
    ]
```

---

## Cache Processing Order

Claude processes components in this order:
```
1. Tools
2. System prompt
3. Messages
```

If you change the system prompt but keep tools the same, you get:
- Cache **read** for tools (saved)
- Cache **write** for the new system prompt (full price)

---

## Checking Cache Performance
```python
usage = response.usage
print(f"Cache write tokens: {usage.cache_creation_input_tokens}")
print(f"Cache read tokens:  {usage.cache_read_input_tokens}")
print(f"Regular tokens:     {usage.input_tokens}")
```

---

## The 1-Hour TTL
```
Request 1  → Cache WRITTEN  → Timer starts (60 min)
Request 2  → Cache HIT      → Timer RESETS (60 min from now)
...
[No requests for 61 minutes]
Request N  → Cache MISS     → Cache WRITTEN again (full price)
```

---

## When to Use / Not Use

| Use caching | Skip caching |
|---|---|
| Large system prompt sent on every request | Short prompts under 1024 tokens |
| Multi-turn Q&A over the same document | Highly dynamic prompts |
| Stable tool schemas on every agentic loop | One-off requests |
| Few-shot examples that don't change | Low-frequency traffic (cache expires before reuse) |

---

## Key Takeaway

> Caching is a **cost and latency optimisation**, not a token reduction.  
> Place breakpoints on the largest, most stable parts of your prompt —
> system prompts and tool schemas are the best starting points.  
> Even a single character change invalidates the cache for that component.

# Files API + Code Execution

---

## Files API

An alternative to base64 encoding — upload a file once, reference it by ID in
any future message.

### Flow
```
1. Upload file → receive FileMetadata(id="file_011CPYfeeBS1CUbF")
2. Reference file_id in messages instead of embedding raw bytes
3. Reuse the same file_id across multiple requests
```

### Using a file in a message (image example)
```python
{
    "type": "image",
    "source": {
        "type": "file",
        "file_id": "file_011CPYfeeBS1CUbF"
    }
}
```

Best for: large files, files referenced multiple times, cleaner message payloads.

---

## Code Execution Tool

A **server-side tool** — no implementation required from you. Just include the
schema and Claude can write and execute Python code autonomously.

### Schema
```python
tools = [{"type": "code_execution_20250522", "name": "code_execution"}]
```

### Execution environment

| Property | Detail |
|---|---|
| Environment | Isolated Docker container |
| Network access | None (no external API calls) |
| Language | Python |
| Executions per response | Multiple (Claude iterates as needed) |

### Response block types
```
Assistant Message → ServerToolUseBlock      (code Claude wrote)
                  → CodeExecutionResultBlock (output from running the code)
                  → TextBlock               (Claude's explanation/summary)
```

---

## Files API + Code Execution — Combined Workflow

Since Docker containers have no network access, the Files API is the primary way
to **get data into and out of** the execution environment.

### Full workflow
```
1. Upload data file (CSV, PDF, etc.) via Files API → get file_id
         |
         v
2. Send user message with:
   - container_upload block (file_id)
   - text block (the task)
         |
         v
3. Claude writes and executes Python code in Docker
         |
         v
4. Claude returns analysis + generated files (plots, reports)
         |
         v
5. Download generated files via Files API using file IDs from response
```

### Code
```python
file_metadata = upload('streaming.csv')

messages = []
add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """Run a detailed analysis to determine major drivers of churn.
            Your final output should include at least one detailed plot."""
        },
        {
            "type": "container_upload",
            "file_id": file_metadata.id      # send file into Docker container
        },
    ],
)

chat(
    messages,
    tools=[{"type": "code_execution_20250522", "name": "code_execution"}]
)
```

### Downloading generated output files
```python
# Generated files (plots, reports) appear as code_execution_output blocks
# Extract the file_id and download
download_file("file_id_from_response")
```

---

## Use Cases

- Data analysis and visualisation (CSV → charts)
- Image processing and manipulation
- Document parsing and transformation
- Mathematical computations
- Report generation with custom formatting

---

## Key Takeaway

> The Files API solves the "no network access" limitation of Docker containers —
> use it to pass data **in** via `container_upload` blocks and retrieve
> generated outputs **out** via file IDs in the response.  
> Together, they let you delegate complex computational tasks entirely to Claude.